# Data Quality Analysis
Author: Aisosa

## 1. Setup
Load libraries and dataset.

In [ ]:
import pandas as pd
import numpy as np

# download the data
df = pd.read_csv("../data/processed/precious_metals_trade_inflation.csv")

# first overview
print(df.shape)
print(df.dtypes)
df.head()

(83765, 12)
country             object
year                 int64
commodity           object
metal_group         object
flow                object
trade_value_usd      int64
weight_kg          float64
category            object
Core CPI           float64
Energy CPI         float64
Food CPI           float64
Headline CPI       float64
dtype: object


,country,year,commodity,metal_group,flow,trade_value_usd,weight_kg,category,Core CPI,Energy CPI,Food CPI,Headline CPI
0,Albania,2013,Precious metal ores and concentrates except si...,Silver,Export,401,29000.0,26_ores_slag_and_ash,0.189061,0.24,4.240740,1.934749
1,Albania,2012,Precious metal ores and concentrates except si...,Silver,Import,668,3890.0,26_ores_slag_and_ash,1.712044,0.90,2.340568,2.027813
2,Albania,2011,Precious metal ores and concentrates except si...,Silver,Import,724,4720.0,26_ores_slag_and_ash,3.248434,2.76,4.754817,3.414678
3,Albania,2009,Precious metal ores and concentrates except si...,Silver,Import,1672,7820.0,26_ores_slag_and_ash,1.568649,1.75,4.911592,2.290375
4,Albania,2009,Precious metal ores and concentrates except si...,Silver,Export,7134,104300.0,26_ores_slag_and_ash,1.568649,1.75,4.911592,2.290375


## 2. Missing Values
Check for null values across all columns.

In [14]:

# missing values
print("Missing values:")
print(df.isnull().sum())

print("missing values percentage:")
print((df.isnull().sum() / len(df)) * 100)

Missing values:
country                0
year                   0
commodity              0
metal_group            0
flow                   0
trade_value_usd        0
weight_kg           4655
category               0
Core CPI           39267
Energy CPI         23024
Food CPI           17663
Headline CPI       16404
dtype: int64
missing values percentage:
country             0.000000
year                0.000000
commodity           0.000000
metal_group         0.000000
flow                0.000000
trade_value_usd     0.000000
weight_kg           5.557214
category            0.000000
Core CPI           46.877574
Energy CPI         27.486420
Food CPI           21.086373
Headline CPI       19.583358
dtype: float64


## 3. Outliers
Check for suspicious values in numerical columns.

In [5]:
#checking outliers
print("numerical statistics")
df[["trade_value_usd", "weight_kg", "Headline CPI"]].describe()

numerical statistics


,trade_value_usd,weight_kg,Headline CPI
count,8.376500e+04,7.911000e+04,67361.000000
mean,7.413464e+07,6.902412e+05,16.748444
std,1.103139e+09,1.546897e+07,145.827381
min,1.000000e+00,0.000000e+00,-72.728996
25%,7.000000e+03,2.000000e+00,1.633060
50%,1.233510e+05,2.150000e+02,2.981579
75%,2.478087e+06,6.156000e+03,6.326885
max,9.696865e+10,1.640361e+09,2947.732910


In [8]:
# how many outliers are there in the trade_value_usd column?
print("entries over 1 Mrd USD:", (df["trade_value_usd"] > 1_000_000_000).sum())
print("entries under 100 USD:", (df["trade_value_usd"] < 100).sum())
print("weight_kg = 0:", (df["weight_kg"] == 0).sum())

# Which countries have the highest trade values?
print("Top 10 Countries based on trade value:")
df.groupby("country")["trade_value_usd"].sum().sort_values(ascending=False).head(10)

entries over 1 Mrd USD: 930
entries under 100 USD: 2779
weight_kg = 0: 16765
Top 10 Countries based on trade value:


country
Hong Kong SAR, China    1395304879570
EU-28                   1216718016081
India                    604339252214
China                    497349292935
Germany                  378285922278
Canada                   358295914468
Italy                    337987924154
Australia                314786436341
Japan                    286227396011
France                   166954014326
Name: trade_value_usd, dtype: int64

## 4. Data Overview
Overview relevant for all chapters.

In [9]:
# Data overview for all chapters
print("Metals Chapter 3")
print(df["metal_group"].value_counts())

print("years available")
print(f"From {df['year'].min()} to {df['year'].max()}")

print("Import vs Export")
print(df["flow"].value_counts())

print("Total countries")
print(f"Number of countries: {df['country'].nunique()}")

print("Switzerland available?")
print(df[df["country"].str.contains("Switz", na=False)]["country"].unique())

Metals Chapter 3
metal_group
Other precious    26320
Gold              26041
Silver            23623
Platinum           7781
Name: count, dtype: int64
years available
From 1988 to 2016
Import vs Export
flow
Import       48416
Export       29801
Re-Export     3793
Re-Import     1755
Name: count, dtype: int64
Total countries
Number of countries: 198
Switzerland available?
['Switzerland']


In [ ]:
print(df["metal_group"].value_counts())
print(df.groupby("metal_group")["trade_value_usd"].sum().sort_values(ascending=False))

## 5. Switzerland Detail
Check if Switzerland has enough data for the Sankey diagram.

In [11]:
# Switzerland detail 
swiss = df[df["country"] == "Switzerland"]

print(len(swiss))
print(swiss["flow"].value_counts())
print(swiss["year"].min(), swiss["year"].max())
print(swiss.groupby("year")["trade_value_usd"].sum().sort_values(ascending=False).head(5))


print(swiss["metal_group"].value_counts())
print(swiss["commodity"].value_counts())

74
flow
Import    43
Export    31
Name: count, dtype: int64
1988 2016
year
2012    10009344
2013     4523374
2011     4295490
2007     4028281
2008     2958027
Name: trade_value_usd, dtype: int64
metal_group
Silver    74
Name: count, dtype: int64
commodity
Precious metal ores and concentrates except silver    48
Silver ores and concentrates                          26
Name: count, dtype: int64


## Data Quality Summary

### Missing Values
- weight_kg: 5.6% missing
- Headline CPI: 19.6% missing
- Core CPI: 46.9% missing — critical!

### Outliers
- 2779 entries under $100 USD — suspicious
- 16765 entries with weight_kg = 0

### Key Findings for the Team
- Dataset only goes until 2016 — 2022 inflation wave not covered!
- Switzerland only has Silver in the dataset — problem for Chapter 4
- Gold dominates trade value with $3.8 trillion